# 03 - Treino e Avaliação

Objetivo: treinar modelos, comparar performance com métricas adequadas para fraude, escolher threshold na validação e medir o teste apenas uma vez.

Métricas principais: PR-AUC, recall, precision, F1 e F-beta.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

from src.config.settings import Settings
from src.data.load_data import RawDataRepository
from src.data.merge_data import FraudDataMerger, first_existing
from src.data.split_data import TemporalSplitter
from src.features.cleaning import FraudDataCleaner
from src.models.evaluate import evaluate_binary_classifier
from src.models.threshold import find_best_threshold
from src.models.train import FraudModelTrainer

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

## 1. Configuração

Use `SAMPLE_ROWS` para acelerar testes no notebook. Para treino completo, deixe `None`.

In [3]:
SAMPLE_ROWS = None  # exemplo: 200_000 para acelerar experimentos
MODEL_NAMES = ["logistic_regression", "random_forest"]

base_settings = Settings(project_root=PROJECT_ROOT)
base_settings.artifacts_dir.mkdir(parents=True, exist_ok=True)
base_settings.artifacts_dir

WindowsPath('D:/developer/workspace_python/financial_transactions_pipeline/artifacts')

## 2. Carregar, unir, limpar e separar por tempo

In [4]:
raw = RawDataRepository(base_settings).load_all()
merged = FraudDataMerger(base_settings).merge(
    transactions=raw["transactions"],
    cards=raw["cards"],
    users=raw["users"],
    mcc_codes=raw["mcc"],
    labels=raw["labels"],
)
cleaned = FraudDataCleaner(base_settings).fit_transform(merged)

if SAMPLE_ROWS is not None:
    time_col = first_existing(cleaned.columns, base_settings.time_column_candidates)
    cleaned = cleaned.sort_values(time_col).head(SAMPLE_ROWS).copy()

splits = TemporalSplitter(base_settings).split(cleaned)

def split_xy(df, settings):
    return df.drop(columns=[settings.target_column]), df[settings.target_column].astype(int)

X_train, y_train = split_xy(splits.train, base_settings)
X_val, y_val = split_xy(splits.validation, base_settings)
X_test, y_test = split_xy(splits.test, base_settings)

len(X_train), len(X_val), len(X_test)

2026-06-13 12:45:52 | INFO | src.data.load_data | Carregando CSV de local: D:\developer\workspace_python\financial_transactions_pipeline\data\raw\transactions_data.csv
2026-06-13 12:46:51 | INFO | src.data.load_data | Carregando CSV de local: D:\developer\workspace_python\financial_transactions_pipeline\data\raw\cards_data.csv
2026-06-13 12:46:51 | INFO | src.data.load_data | Carregando CSV de local: D:\developer\workspace_python\financial_transactions_pipeline\data\raw\users_data.csv
2026-06-13 12:46:51 | INFO | src.data.load_data | Carregando JSON de local: D:\developer\workspace_python\financial_transactions_pipeline\data\raw\mcc_codes.json
2026-06-13 12:46:51 | INFO | src.data.load_data | Carregando JSON de local: D:\developer\workspace_python\financial_transactions_pipeline\data\raw\train_fraud_labels.json
2026-06-13 12:48:40 | INFO | src.data.merge_data | Dataset consolidado: 8914963 linhas, 39 colunas


D:\developer\workspace_python\financial_transactions_pipeline\src\features\cleaning.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column], errors="coerce")
D:\developer\workspace_python\financial_transactions_pipeline\src\features\cleaning.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column], errors="coerce")


(6240474, 1337244, 1337245)

## 3. Comparar modelos

A escolha do threshold é feita apenas com a validação. O teste não participa da otimização.

In [ ]:
results = []
fitted_models = {}

for model_name in MODEL_NAMES:
    settings = Settings(project_root=PROJECT_ROOT, model_name=model_name)
    pipeline = FraudModelTrainer(settings).train(X_train, y_train)
    val_scores = pipeline.predict_proba(X_val)[:, 1]
    threshold, threshold_metrics = find_best_threshold(
        y_val.to_numpy(),
        val_scores,
        beta=settings.threshold_beta,
        min_precision=settings.min_precision_for_threshold,
    )
    val_metrics = evaluate_binary_classifier(y_val.to_numpy(), val_scores, threshold, settings.threshold_beta)
    test_scores = pipeline.predict_proba(X_test)[:, 1]
    test_metrics = evaluate_binary_classifier(y_test.to_numpy(), test_scores, threshold, settings.threshold_beta)

    results.append({"model": model_name, "split": "validation", **val_metrics})
    results.append({"model": model_name, "split": "test", **test_metrics})
    fitted_models[model_name] = {"pipeline": pipeline, "threshold": threshold, "val_scores": val_scores, "test_scores": test_scores}

metrics_df = pd.DataFrame(results)
metrics_df.sort_values(["split", "pr_auc"], ascending=[True, False])

2026-06-13 12:55:19 | INFO | src.models.train | Treinando modelo logistic_regression com 6240474 linhas


In [ ]:
metric_cols = ["pr_auc", "recall", "precision", "f1", "fbeta"]
display_df = metrics_df.melt(id_vars=["model", "split"], value_vars=metric_cols, var_name="metric", value_name="value")

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=display_df, x="metric", y="value", hue="model", ax=ax)
ax.set_title("Comparação de métricas por modelo")
ax.set_ylim(0, 1)
plt.show()

## 4. Selecionar melhor modelo por PR-AUC de validação

In [ ]:
best_model_name = (
    metrics_df.query("split == 'validation'")
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["model"]
)
best = fitted_models[best_model_name]
best_model_name, best["threshold"]

## 5. Curvas PR e ROC no teste

Para fraude, a curva Precision-Recall costuma ser mais informativa que ROC quando a classe positiva é rara.

In [ ]:
test_scores = best["test_scores"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
PrecisionRecallDisplay.from_predictions(y_test, test_scores, ax=axes[0])
RocCurveDisplay.from_predictions(y_test, test_scores, ax=axes[1])
axes[0].set_title(f"Precision-Recall - {best_model_name}")
axes[1].set_title(f"ROC - {best_model_name}")
plt.tight_layout()
plt.show()

## 6. Matriz de confusão no threshold escolhido

In [ ]:
test_pred = (test_scores >= best["threshold"]).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, labels=[0, 1], cmap="Blues")
plt.title(f"Matriz de confusão - {best_model_name}")
plt.show()

## 7. Salvar melhor pipeline e metadados

Esta célula salva uma pipeline compatível com o serviço batch e a API.

In [ ]:
best_pipeline_path = base_settings.pipeline_path
best_metadata_path = base_settings.metadata_path

joblib.dump(best["pipeline"], best_pipeline_path)
joblib.dump(
    {
        "model_name": best_model_name,
        "threshold": best["threshold"],
        "validation_metrics": metrics_df.query("model == @best_model_name and split == 'validation'").iloc[0].to_dict(),
        "test_metrics": metrics_df.query("model == @best_model_name and split == 'test'").iloc[0].to_dict(),
    },
    best_metadata_path,
)

best_pipeline_path, best_metadata_path